# Problem 1

Reads mapping against IS do not map against genome in which IS are masked. If not masked, reads only map to internal part of IS6100.

Answer: minimap2 doesn't get the small alignments (<32). So in the step where reads are mapped against the IS sequence, only reads with the larger part mapping to the IS are reported. When mapping these against the genome, the flanking part is too short to produce an alignment. 

In [ ]:
%%bash

minimap2 -ax sr -k 7 -w 5  -t 4 \
  /home/cristobal/TB/projects/detettoreTB/detettore/tmp/ref_masked.fasta \
  /home/cristobal/TB/projects/detettoreTB/detettore/tmp/partial_hits.fastq.gz

In [ ]:
%%bash

minimap2 -ax sr -k 9 -w 5  -t 4 \
  /home/cristobal/TB/projects/detettoreTB/detettore/testdata/reference/H37Rv.fasta \
  /home/cristobal/TB/projects/detettoreTB/detettore/tmp/partial_hits.fastq.gz

What about bwa mem?

In [ ]:
%%bash

bwa index testdata/reference/H37Rv.fasta

bwa mem -k 7 -a \
  testdata/reference/H37Rv.fasta \
  /home/cristobal/TB/projects/detettoreTB/detettore/tmp/partial_hits.fastq.gz \
  | samtools view -@ 4 -hu -o tmp/aln.raw.bam 

samtools sort -@ 4 tmp/aln.raw.bam -o tmp/aln.bam

samtools index tmp/aln.bam


Is it because we select reads where the larger part maps to IS rather than the flanking region?

In [ ]:
import pandas

paf = pandas.read_csv("/home/cristobal/TB/projects/detettoreTB/detettore/tmp/reads_vs_is.paf", sep="\t", header=None)
paf[10].describe()

In [ ]:
import seaborn as sns

partial_hits = paf[paf[10] < 50]

sns.displot(partial_hits[10], kde=True)

## Solution

Use blastn to get reads that map against IS? Takes too long.

Better: ignore 50bp SE reads. 

# Paired-end reads

In [ ]:
%%bash

minimap2 -x sr -Yc \
  /home/cristobal/TB/projects/detettoreTB/detettore/testdata/is_targets/IS6110.fasta \
  /home/cristobal/TB/projects/detettoreTB/detettore/testdata/reads/RW-TB008.PE_l150m200s10f40_1.fq.gz \
  /home/cristobal/TB/projects/detettoreTB/detettore/testdata/reads/RW-TB008.PE_l150m200s10f40_2.fq.gz